# Prefacio y Arquitectura

In [4]:
#!/usr/bin/env python
# coding: utf-8
"""
entorno_importacion.py
======================

Capa de adquisición, normalización y depuración de datos de fondos de inversión.
Diseñada como módulo integrable con el motor de backtesting del proyecto.

Contrato de salida hacia el motor
---------------------------------
- `EntornoImportacion.get_engine_data()` -> Dict[ISIN, pd.DataFrame]
    Cada DataFrame está indexado por DatetimeIndex (ordenado, sin duplicados)
    y contiene estrictamente las columnas Open, High, Low, Close, Volume.

- `EntornoImportacion.get_universe()` -> pd.DataFrame
    Metadatos del universo: ISIN, ticker, nombre, categoría, fuente.

- `EngineDataFeed`
    Wrapper iterable compatible con motores que consumen datos barra a barra.
"""

from __future__ import annotations

import csv
import logging
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Tuple

import numpy as np
import pandas as pd

# --- Dependencias opcionales (instalación automática) -----------------------
try:
    import yfinance as yf
    from sqlalchemy import create_engine
except ImportError:
    print("Instalando dependencias necesarias (yfinance, sqlalchemy)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance", "sqlalchemy"])
    import yfinance as yf
    from sqlalchemy import create_engine


# --- Universo por defecto ---------------------------------------------------
DEFAULT_UNIVERSE: Dict[str, Dict[str, Dict[str, str]]] = {
    "Monetario": {
        "Renta 4 Renta Fija 6 Meses, Fi": {"ISIN": "ES0128520006", "ticker": "0P0000KY8J.F"},
        "Renta 4 Renta Fija, Fi Clase R": {"ISIN": "ES0176954008", "ticker": "0P0000YLAY.F"},
        "Vanguard Global Short-Term Bond EUR Hedged Acc": {"ISIN": "IE00BH65QP47", "ticker": "0P00012NJH.F"},
        "Vanguard Global Bond Index EUR Hedged Acc": {"ISIN": "IE00B18GC888", "ticker": "0P00012I69.F"},
        "DWS EO Ultra Short Fix.Income I": {"ISIN": "LU0080237943", "ticker": "DI4C.F"},
    },
    "Renta fija": {
        "Vanguard Euro Government Bond UCITS ETF": {"ISIN": "IE0007472990", "ticker": "VGEB"},
    },
    "Renta variable": {
        "Vanguard Global Stock Index Fund EUR Acc": {"ISIN": "IE00B03HD191", "ticker": "0P00000WLG.F"},
        "Vanguard Emerging Markets Stock Index Inv EUR Acc": {"ISIN": "IE0031786696", "ticker": "0P00012I6A.F"},
        "Vanguard Global Stock Index Inv EUR Acc": {"ISIN": "IE00B03HCZ61", "ticker": "0P00000RQC.F"},
        "Amundi MSCI Emerging Markets Swap UCITS ETF EUR Acc": {"ISIN": "LU1681045370", "ticker": "AEEM.PA"},
        "Renta Variable Global Small Caps": {"ISIN": "IE00B42W4L06", "ticker": "0P0000XR9M.F"},
        "Vanguard Emerging Markets Stock Index Fund EUR Acc": {"ISIN": "IE0031786142", "ticker": "0P000060MS.F"},
    },
}


# --- Configuración y Feed para el Motor -------------------------------------
@dataclass
class EngineConfig:
    """Configuración que el motor espera recibir del módulo de datos."""
    fecha_inicio: Optional[pd.Timestamp] = None
    fecha_fin: Optional[pd.Timestamp] = None
    columnas_ohlcv: Tuple[str, ...] = ("Open", "High", "Low", "Close", "Volume")
    timezone: str = "UTC"
    manejo_duplicados: str = "last"  # 'last' | 'first' | 'mean'
    interpolar_huecos: bool = True

    def __post_init__(self):
        if isinstance(self.fecha_inicio, str):
            self.fecha_inicio = pd.Timestamp(self.fecha_inicio)
        if isinstance(self.fecha_fin, str):
            self.fecha_fin = pd.Timestamp(self.fecha_fin)


@dataclass
class EngineDataFeed:
    """
    Feed iterable barra-a-barra para el motor.
    Optimizado con pre-alineación de índices para búsqueda O(1).
    """
    data: Dict[str, pd.DataFrame]
    config: EngineConfig

    def __post_init__(self):
        if not self.data:
            self._index = pd.DatetimeIndex([])
            self._aligned_data = {}
            return
        
        # Unión robusta de índices (compatible con todas las versiones de pandas)
        self._index = pd.concat([df[[]] for df in self.data.values()], axis=1).index.sort_values()
        
        # Pre-alinear datos para evitar búsquedas costosas durante la iteración
        self._aligned_data = {
            isin: df.reindex(self._index) 
            for isin, df in self.data.items()
        }

    def __iter__(self) -> Iterator[Tuple[pd.Timestamp, Dict[str, pd.Series]]]:
        for ts in self._index:
            snapshot = {}
            for isin, df in self._aligned_data.items():
                row = df.loc[ts]
                # Verificamos que haya al menos un dato válido (ej. Close)
                if pd.notna(row.get("Close")):
                    snapshot[isin] = row
            if snapshot:
                yield ts, snapshot

    def __len__(self) -> int:
        return len(self._index)

    @property
    def index(self) -> pd.DatetimeIndex:
        return self._index

    def get_history(self, isin: str) -> pd.DataFrame:
        """Devuelve la serie completa de un ISIN (acceso directo para el motor)."""
        return self.data.get(isin, pd.DataFrame())


# --- Clase Principal --------------------------------------------------------
class EntornoImportacion:
    """Orquestador de adquisición, normalización y depuración de datos."""

    FUENTES_AGRESIVAS = {"yahoo", "investing"}
    ORDEN_PRIORIDAD_FUENTE = ["gestora", "r4", "myinvestor", "investing", "yahoo"]

    def __init__(
        self,
        diccionario_datos: Dict,
        fecha_inicio: str = "2010-01-01",
        fecha_fin: Optional[str] = None,
        dir_base: str = ".",
        config_motor: Optional[EngineConfig] = None,
    ) -> None:
        if not diccionario_datos or not isinstance(diccionario_datos, dict):
            raise ValueError("Se debe aportar un 'diccionario_datos' válido.")

        self.fecha_inicio = pd.to_datetime(fecha_inicio)
        self.fecha_fin = pd.to_datetime(fecha_fin) if fecha_fin else None
        self.dir_base = Path(dir_base)

        # --- Carpetas de datos
        self.dir_datos = self.dir_base / "Datos_fondos_csv"
        self.dir_datos.mkdir(parents=True, exist_ok=True)

        self.dirs_fuentes: Dict[str, Path] = {}
        for fuente in self.ORDEN_PRIORIDAD_FUENTE:
            carpeta = self.dir_base / f"datos_fi_{fuente}"
            carpeta.mkdir(parents=True, exist_ok=True)
            self.dirs_fuentes[fuente] = carpeta

        self.dir_backup = self.dir_datos / "_originales"
        self.dir_backup.mkdir(parents=True, exist_ok=True)

        self.ruta_db = self.dir_datos / "san_database.db"
        self.db_engine = create_engine(f"sqlite:///{self.ruta_db}", echo=False)

        # --- Outputs del motor
        self.dir_outputs = self.dir_base / "outputs"
        self.dir_logs = self.dir_base / "logs"
        for d in (
            self.dir_outputs / "backtest",
            self.dir_outputs / "figures" / "png",
            self.dir_outputs / "figures" / "pdf",
            self.dir_outputs / "reports" / "pdf",
            self.dir_logs,
        ):
            d.mkdir(parents=True, exist_ok=True)

        self.logger = self._configurar_logger()

        # --- Estado
        self.valores_seleccionados = self._procesar_diccionario(diccionario_datos)
        self.datos_cargados: Dict[str, pd.DataFrame] = {}
        self.fuente_datos: Dict[str, str] = {}
        
        # Sincronizar configuración
        if config_motor:
            if config_motor.fecha_inicio:
                self.fecha_inicio = pd.to_datetime(config_motor.fecha_inicio)
            if config_motor.fecha_fin:
                self.fecha_fin = pd.to_datetime(config_motor.fecha_fin)
            self.config_motor = config_motor
        else:
            self.config_motor = EngineConfig(fecha_inicio=self.fecha_inicio, fecha_fin=self.fecha_fin)

        self._imprimir_resumen()

    def _configurar_logger(self) -> logging.Logger:
        logger = logging.getLogger(f"EntornoImportacion.{id(self)}")
        logger.setLevel(logging.INFO)
        logger.handlers.clear()

        fh = logging.FileHandler(
            self.dir_logs / f"datos_{time.strftime('%Y%m%d_%H%M%S')}.log", encoding="utf-8"
        )
        fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
        fh.setFormatter(fmt)
        logger.addHandler(fh)

        ch = logging.StreamHandler(sys.stdout)
        ch.setFormatter(fmt)
        logger.addHandler(ch)
        return logger

    def _procesar_diccionario(self, diccionario_datos: Dict) -> pd.DataFrame:
        datos_planos = []
        for categoria, fondos in diccionario_datos.items():
            for nombre, info in fondos.items():
                if isinstance(info, dict) and "ISIN" in info and "ticker" in info:
                    datos_planos.append({
                        "categoria": categoria, "nombre": nombre,
                        "ISIN": str(info["ISIN"]).strip(), "ticker": str(info["ticker"]).strip(),
                    })
        return pd.DataFrame(datos_planos)

    def _imprimir_resumen(self) -> None:
        n_valores = len(self.valores_seleccionados)
        fin_str = self.fecha_fin.strftime("%Y-%m-%d") if self.fecha_fin else "Actualidad"
        self.logger.info("CONFIGURACIÓN ENTORNO COMPLETADA")
        self.logger.info("Nº valores en diccionario: %d", n_valores)
        self.logger.info("Rango fechas: %s -> %s", self.fecha_inicio.strftime("%Y-%m-%d"), fin_str)
        self.logger.info("Ruta datos normalizados: %s", self.dir_datos.resolve())

    def _prioridad_fuente(self, fuente: str) -> int:
        try:
            return self.ORDEN_PRIORIDAD_FUENTE.index(fuente)
        except ValueError:
            return len(self.ORDEN_PRIORIDAD_FUENTE)

    def _limpiar_numero_universal(self, serie: pd.Series) -> pd.Series:
        def limpiar(v):
            v = str(v).strip()
            if v in ("", "nan", "None"):
                return np.nan
            if "," in v:
                v = v.replace(".", "").replace(",", ".")
            elif v.count(".") > 1:
                partes = v.split(".")
                v = "".join(partes[:-1]) + "." + partes[-1]
            return v
        return serie.apply(limpiar)

    def _parse_vliq_r4(self, filepath: Path) -> pd.DataFrame:
        rows = []
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            reader = csv.reader(f, delimiter=";")
            for row in reader:
                if len(row) < 2:
                    continue
                for i in range(len(row) - 1):
                    date_str = row[i].strip()
                    if not re.match(r"\d{2}/\d{2}/\d{4}", date_str):
                        continue
                    val_str = row[i + 1].strip()
                    if "|" in val_str:
                        val_str = val_str.replace("|", "").replace(" ", "")
                    else:
                        val_str = re.sub(r"[^0-9.,-]", "", val_str)
                        if "," in val_str and "." in val_str:
                            val_str = val_str.replace(".", "").replace(",", ".")
                        elif "," in val_str:
                            val_str = val_str.replace(",", ".")
                    try:
                        date = pd.to_datetime(date_str, format="%d/%m/%Y")
                        val = float(val_str)
                        rows.append((date, val))
                        break
                    except ValueError:
                        continue
        if not rows:
            return pd.DataFrame()
        df = pd.DataFrame(rows, columns=["Date", "Close"]).set_index("Date")
        df = df[~df.index.duplicated(keep="last")].sort_index()
        df["Open"] = df["High"] = df["Low"] = df["Close"]
        df["Volume"] = 0.0
        return df

    def _load_csv_robusto(self, filepath: Path) -> pd.DataFrame:
        filepath = Path(filepath)
        for sep in [",", ";", "\t"]:
            for enc in ["utf-8", "latin-1"]:
                try:
                    df = pd.read_csv(filepath, sep=sep, encoding=enc, dtype=str, on_bad_lines="skip", engine="c")
                    if len(df.columns) > 1:
                        date_col = next((c for c in df.columns if re.search(r"date|fecha", c, re.I)), df.columns[0])
                        
                        # Limpieza robusta de fechas
                        df[date_col] = (
                            df[date_col].astype(str)
                            .str.replace(r"[;\s]+", " ", regex=True)
                            .str.split(" ").str[0]
                            .str.strip()
                        )
                        df[date_col] = pd.to_datetime(df[date_col], errors="coerce", format="mixed", dayfirst=True)
                        df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()

                        close_col = next(
                            (c for c in df.columns if re.search(r"close|nav|valor|precio|ultimo|último|adj close", c, re.I)),
                            df.columns[1],
                        )
                        if close_col != "Close":
                            df.rename(columns={close_col: "Close"}, inplace=True)

                        df["Close"] = self._limpiar_numero_universal(df["Close"])
                        df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
                        df = df.dropna(subset=["Close"])

                        if not df.empty:
                            for col in ["Open", "High", "Low"]:
                                if col not in df.columns:
                                    df[col] = df["Close"]
                            if "Volume" not in df.columns:
                                df["Volume"] = 0.0

                            if self.fecha_inicio is not None:
                                df = df[df.index >= self.fecha_inicio]
                            if self.fecha_fin is not None:
                                df = df[df.index <= self.fecha_fin]

                            return df[["Open", "High", "Low", "Close", "Volume"]]
                except Exception:
                    continue

        df_r4 = self._parse_vliq_r4(filepath)
        if not df_r4.empty:
            if self.fecha_inicio is not None:
                df_r4 = df_r4[df_r4.index >= self.fecha_inicio]
            if self.fecha_fin is not None:
                df_r4 = df_r4[df_r4.index <= self.fecha_fin]
        return df_r4

    def descargar_y_cargar_precios(self, forzar_descarga: bool = False) -> None:
        self.logger.info("Modo: %s", "Forzar descarga (Yahoo)" if forzar_descarga else "Cargar local")
        for _, row in self.valores_seleccionados.iterrows():
            isin, ticker = row["ISIN"], row["ticker"]
            if not ticker:
                continue
            ruta_csv = self.dir_datos / f"{isin}.csv"
            fuente_descarga = None

            if not forzar_descarga and ruta_csv.exists():
                df = self._load_csv_robusto(ruta_csv)
            else:
                try:
                    df_yf = yf.download(
                        ticker,
                        start=self.fecha_inicio.strftime("%Y-%m-%d"),
                        end=self.fecha_fin.strftime("%Y-%m-%d") if self.fecha_fin else None,
                        progress=False,
                        auto_adjust=True,
                    )
                    if df_yf.empty:
                        continue
                    if isinstance(df_yf.columns, pd.MultiIndex):
                        df_yf.columns = df_yf.columns.get_level_values(0)
                    df_yf.index = pd.to_datetime(df_yf.index)
                    df_yf.index.name = "Date"
                    if "Close" not in df_yf.columns and "Adj Close" in df_yf.columns:
                        df_yf.rename(columns={"Adj Close": "Close"}, inplace=True)
                    df_yf.to_csv(ruta_csv)
                    df = df_yf
                    fuente_descarga = "yahoo"
                except Exception as exc:
                    self.logger.warning("Fallo descarga %s (%s): %s", isin, ticker, exc)
                    continue

            if not df.empty and "Close" in df.columns:
                df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
                df.dropna(subset=["Close"], inplace=True)
                self.datos_cargados[isin] = df
                if fuente_descarga:
                    self.fuente_datos[isin] = fuente_descarga
                elif isin not in self.fuente_datos:
                    self.fuente_datos[isin] = "yahoo"

    def importar_csv_externos(self) -> None:
        codigos_sin_isin = {"EP2", "F1467"}
        total_importados = 0
        conflictos: List[Tuple[str, str, str]] = []

        for fuente in self.ORDEN_PRIORIDAD_FUENTE:
            carpeta = self.dirs_fuentes[fuente]
            archivos = list(carpeta.glob("*.csv"))
            if not archivos:
                continue

            self.logger.info("Importando '%s' (%d) desde %s", fuente.upper(), len(archivos), carpeta.name)
            for f in archivos:
                stem = f.stem.replace("vliq_", "").strip()
                for sufijo in ("_r4", "_investing", "_myinvestor", "_yahoo", "_gestora"):
                    if stem.lower().endswith(sufijo):
                        stem = stem[: -len(sufijo)]
                        break
                while stem.lower().endswith(".csv"):
                    stem = stem[:-4]
                isin = stem

                es_isin_valido = bool(re.match(r"^[A-Z]{2}[A-Z0-9]{9}\d$", isin))
                if not es_isin_valido and isin not in codigos_sin_isin:
                    continue

                if isin in self.fuente_datos:
                    fuente_existente = self.fuente_datos[isin]
                    if self._prioridad_fuente(fuente_existente) <= self._prioridad_fuente(fuente):
                        conflictos.append((isin, fuente_existente, fuente))
                        continue

                df = self._load_csv_robusto(f)
                if not df.empty:
                    backup_path = self.dir_backup / f"{fuente}_{f.name}"
                    if not backup_path.exists():
                        shutil.copy2(f, backup_path)
                    path_destino = self.dir_datos / f"{isin}.csv"
                    df.index.name = "Date"
                    df.to_csv(path_destino, sep=",", decimal=".", date_format="%Y-%m-%d")
                    self.fuente_datos[isin] = fuente
                    total_importados += 1
                    self.logger.info("Guardado: %s.csv", isin)

        self.logger.info("%d ficheros importados a '%s'.", total_importados, self.dir_datos.name)
        if conflictos:
            self.logger.warning("%d conflicto(s) de ISIN resueltos por prioridad.", len(conflictos))

    def normalizar_csv_directorio(self) -> None:
        csvs = sorted(self.dir_datos.glob("*.csv"))
        if not csvs:
            return
        self.logger.info("Normalizando %d archivos CSV", len(csvs))
        for ruta in csvs:
            if ruta.parent == self.dir_backup:
                continue
            df = self._load_csv_robusto(ruta)
            if not df.empty:
                df.index.name = "Date"
                df = df[~df.index.duplicated(keep="last")].sort_index()
                df.to_csv(ruta, sep=",", decimal=".", date_format="%Y-%m-%d")

    def depurar_datos(
        self,
        umbral_generico: float = 0.15,
        umbrales_por_categoria: Optional[Dict[str, float]] = None,
        min_dias_planos: int = 5,
        verbose: bool = True,
    ) -> pd.DataFrame:
        reporte = []
        mapeo_categoria = (
            self.valores_seleccionados.set_index("ISIN")["categoria"].to_dict()
            if "categoria" in self.valores_seleccionados.columns else {}
        )

        for isin, df in self.datos_cargados.items():
            if "Close" not in df.columns or df.empty:
                continue
            df = df.sort_index().copy()
            n_original = len(df)
            fuente = self.fuente_datos.get(isin, "desconocida")
            es_agresiva = fuente in self.FUENTES_AGRESIVAS

            df["Close"] = df["Close"].replace(0, np.nan)

            racha = 0
            if es_agresiva:
                precios = df["Close"].ffill()
                diffs = precios.diff().iloc[::-1]
                for igual in diffs:
                    if igual == 0:
                        racha += 1
                    else:
                        break
                if racha >= min_dias_planos:
                    df = df.iloc[:-racha] if racha > 0 else df

            categoria = mapeo_categoria.get(isin)
            umbral_base = (umbrales_por_categoria or {}).get(categoria, umbral_generico)
            umbral = umbral_base * 1.5 if not es_agresiva else umbral_base

            rets = df["Close"].pct_change()
            outliers = rets.abs() > umbral
            n_outliers = int(outliers.sum())
            if n_outliers:
                df.loc[outliers, "Close"] = np.nan

            df["Close"] = df["Close"].interpolate(method="time").ffill().bfill()

            for col in ["Open", "High", "Low"]:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                    if df[col].equals(df["Close"]):
                        df[col] = df["Close"]

            self.datos_cargados[isin] = df
            reporte.append({
                "ISIN": isin, "fuente": fuente, "categoria": categoria,
                "dias_originales": n_original, "dias_finales": len(df),
                "cola_cortada": racha, "outliers_corregidos": n_outliers,
                "umbral_usado": round(umbral, 4),
            })

        reporte_df = pd.DataFrame(reporte)
        if verbose and not reporte_df.empty:
            self.logger.info("Limpieza profunda finalizada:\n%s", reporte_df.to_string(index=False))
        return reporte_df

    def seleccionar_activos(self) -> List[str]:
        csvs_disponibles = sorted(self.dir_datos.glob("*.csv"))
        isins_disponibles = [p.stem for p in csvs_disponibles]
        if not isins_disponibles:
            return []
        mapeo_info = self.valores_seleccionados.set_index("ISIN")["nombre"].to_dict()
        for i, isin in enumerate(isins_disponibles, 1):
            print(f"   {i:>2}. {mapeo_info.get(isin, 'Externo')} — ISIN: {isin}")
        sel = input("\nNúmeros separados por coma o 'todos': ").strip()
        if sel.lower() == "todos":
            return isins_disponibles
        try:
            return [isins_disponibles[int(x) - 1] for x in sel.split(",")]
        except Exception:
            return []

    def cargar_seleccion(self, isins: Iterable[str]) -> None:
        for isin in isins:
            df = self._load_csv_robusto(self.dir_datos / f"{isin}.csv")
            if not df.empty:
                self.datos_cargados[isin] = df

    def cargar_todo(self) -> None:
        """Carga en memoria todos los CSVs disponibles (no interactivo)."""
        isins = [p.stem for p in sorted(self.dir_datos.glob("*.csv"))]
        self.cargar_seleccion(isins)

    # =========================================================================
    # API PARA EL MOTOR
    # =========================================================================
    def get_universe(self) -> pd.DataFrame:
        """Devuelve los metadatos del universo de activos."""
        df = self.valores_seleccionados.copy()
        df["fuente"] = df["ISIN"].map(self.fuente_datos)
        return df

    def get_engine_data(self) -> Dict[str, pd.DataFrame]:
        """
        Devuelve los DataFrames ya depurados listos para el motor.
        Garantiza: índice DatetimeIndex monotónico creciente, sin duplicados,
        columnas en orden OHLCV, dtype float64.
        """
        salida: Dict[str, pd.DataFrame] = {}
        cols_ohlcv = list(self.config_motor.columnas_ohlcv)
        
        for isin, df in self.datos_cargados.items():
            if df is None or df.empty:
                continue
            
            df = df.copy()
            df.index = pd.to_datetime(df.index)
            
            # 1. Manejo de duplicados
            keep = self.config_motor.manejo_duplicados
            df = df[~df.index.duplicated(keep=keep)]
            df = df.sort_index()
            
            # 2. Garantizar columnas OHLCV
            for col in cols_ohlcv:
                if col not in df.columns:
                    df[col] = df["Close"] if "Close" in df.columns else np.nan
            
            # 3. Interpolación de huecos
            if self.config_motor.interpolar_huecos and "Close" in df.columns:
                df["Close"] = df["Close"].interpolate(method="time").ffill().bfill()
                for col in ("Open", "High", "Low"):
                    if col in df.columns:
                        df[col] = df[col].where(df[col].notna(), df["Close"])
                if "Volume" in df.columns:
                    df["Volume"] = df["Volume"].fillna(0.0)
            
            # 4. Reordenar y tipificar estrictamente
            df = df[cols_ohlcv].astype("float64")
            salida[isin] = df
            
        return salida

    def to_engine_format(self) -> EngineDataFeed:
        """Construye un `EngineDataFeed` iterable barra-a-barra."""
        data = self.get_engine_data()
        return EngineDataFeed(data=data, config=self.config_motor)

    def snapshot(self) -> Dict[str, object]:
        """Resumen serializable del estado actual."""
        return {
            "fecha_inicio": str(self.fecha_inicio.date()),
            "fecha_fin": str(self.fecha_fin.date()) if self.fecha_fin else None,
            "n_activos_diccionario": len(self.valores_seleccionados),
            "n_activos_cargados": len(self.datos_cargados),
            "isins_cargados": list(self.datos_cargados.keys()),
            "fuentes": dict(self.fuente_datos),
        }

    def menu(self) -> None:
        while True:
            print("\n" + "=" * 55)
            print(" SISTEMA: IMPORTAR · NORMALIZAR · DEPURAR · SELECCIONAR")
            print("=" * 55)
            print("1. Descargar/Actualizar cotizaciones (Yahoo Finance)")
            print("2. Importar CSVs por fuente (r4, myinvestor, etc.)")
            print("3. Normalizar CSVs del directorio")
            print("4. Seleccionar y cargar activos en memoria")
            print("5. Depurar datos en memoria")
            print("6. Exportar a formato motor (EngineDataFeed)")
            print("7. Salir")
            print("=" * 55)

            opcion = input("Selecciona una opción (1-7): ").strip()
            if opcion == "1":
                self.descargar_y_cargar_precios(input("¿Forzar? (s/n): ").lower() == "s")
            elif opcion == "2":
                self.importar_csv_externos()
            elif opcion == "3":
                self.normalizar_csv_directorio()
            elif opcion == "4":
                isins = self.seleccionar_activos()
                if isins:
                    self.cargar_seleccion(isins)
            elif opcion == "5":
                self.depurar_datos()
            elif opcion == "6":
                feed = self.to_engine_format()
                print(f"✅ EngineDataFeed con {len(feed)} barras y {len(feed.data)} activos.")
            elif opcion == "7":
                print("Cerrando sistema.")
                break
            else:
                print("❌ Opción no válida.")

            if opcion != "7":
                input("\n✅ Tarea finalizada. Presiona Enter para continuar...")


# ===========================================================================
# Punto de entrada y ejemplo de integración
# ===========================================================================
if __name__ == "__main__":
    # 1. Configuración del motor
    config = EngineConfig(
        fecha_inicio=pd.Timestamp("2018-01-01"),
        fecha_fin=pd.Timestamp("2026-01-01"),
        manejo_duplicados="last",
        interpolar_huecos=True,
    )

    # 2. Inicializar entorno
    entorno = EntornoImportacion(
        diccionario_datos=DEFAULT_UNIVERSE,
        dir_base=".",
        config_motor=config,
    )

    # 3. Pipeline completo
    entorno.descargar_y_cargar_precios(forzar_descarga=False)
    entorno.importar_csv_externos()
    entorno.normalizar_csv_directorio()
    entorno.cargar_todo()
    entorno.depurar_datos()

    # 4. Acceso directo por ISIN
    datos = entorno.get_engine_data()
    if "IE00B03HD191" in datos:
        df = datos["IE00B03HD191"]
        print(f"\n📈 Vanguard Global Stock: {len(df)} barras")
        print(df.tail(3))

    # 5. Iteración barra-a-barra (estilo event-driven)
    print("\n🔄 Primeras 5 barras del feed:")
    feed = entorno.to_engine_format()
    
    # Motor ficticio para demostrar la integración sin errores
    class MotorFicticio:
        def on_bar(self, ts: pd.Timestamp, snapshot: dict):
            print(f"  {ts.strftime('%Y-%m-%d')} → {len(snapshot)} activos con datos")

    motor = MotorFicticio()
    for i, (ts, snapshot) in enumerate(feed):
        if i >= 5:
            break
        motor.on_bar(ts, snapshot)

    # 6. Metadatos del universo
    universe_df = entorno.get_universe()
    print(f"\n📋 Universo: {len(universe_df)} activos en {universe_df['categoria'].nunique()} categorías")

2026-08-31 00:17:29 | INFO    | CONFIGURACIÓN ENTORNO COMPLETADA
2026-08-31 00:17:29 | INFO    | Nº valores en diccionario: 12
2026-08-31 00:17:29 | INFO    | Rango fechas: 2018-01-01 -> 2026-01-01
2026-08-31 00:17:29 | INFO    | Ruta datos normalizados: /media/enri/MEMORIA USB/@Libro_Estrategias/notebooks/Datos_fondos_csv
2026-08-31 00:17:29 | INFO    | Modo: Cargar local


$DI4C.F: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-01-01)

1 Failed download:
['DI4C.F']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-01-01)
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: VGEB"}}}
$VGEB: possibly delisted; no timezone found

1 Failed download:
['VGEB']: possibly delisted; no timezone found


2026-08-31 00:17:31 | INFO    | 0 ficheros importados a 'Datos_fondos_csv'.
2026-08-31 00:17:31 | INFO    | Normalizando 10 archivos CSV
2026-08-31 00:17:32 | INFO    | Limpieza profunda finalizada:
        ISIN fuente      categoria  dias_originales  dias_finales  cola_cortada  outliers_corregidos  umbral_usado
ES0128520006  yahoo      Monetario             1946          1946             0                    0          0.15
ES0176954008  yahoo      Monetario             1945          1945             0                    0          0.15
IE00BH65QP47  yahoo      Monetario             2026          2026             0                    0          0.15
IE00B18GC888  yahoo      Monetario             2026          2026             0                    0          0.15
IE00B03HD191  yahoo Renta variable             2026          2026             0                    0          0.15
IE0031786696  yahoo Renta variable             2026          2026             0                    0          0